# 01 — Data Ingestion Pipeline

**AI Equity Research Lab** — FGV EAESP (Aulas 1–2)

This notebook runs the full ingestion pipeline:
1. **CVM DFP** — balance sheet, income statement, cash flow (2020–2024)
2. **Market data** — daily prices and returns via Yahoo Finance (2020–2025)
3. **Macro data** — Selic, IPCA, USD/BRL from BCB/SGS (2020–2025)

### Universe
| Sector | Tickers |
|---|---|
| Bancos | ITUB4, BBDC4, BBAS3, SANB11, ABCB4 |
| Energia Elétrica | EGIE3, EQTL3, CPFE3, TAEE11, CMIG4 |

In [ ]:
import sys
from pathlib import Path

# Add project root to path so we can import src modules
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

print(f"Project root: {PROJECT_ROOT}")

## 1. CVM DFP Ingestion

In [ ]:
from src.ingest.cvm import run as run_cvm

fundamentals, income = run_cvm()

In [ ]:
# Inspect fundamentals_long
fundamentals = pd.read_parquet(PROCESSED_DIR / "fundamentals_long.parquet")
print(f"Shape: {fundamentals.shape}")
print(f"\nDtypes:\n{fundamentals.dtypes}")
print(f"\nTickers: {sorted(fundamentals['ticker'].unique())}")
print(f"Years: {sorted(fundamentals['year'].unique())}")
fundamentals.head(5)

In [ ]:
# Inspect income_long
income = pd.read_parquet(PROCESSED_DIR / "income_long.parquet")
print(f"Shape: {income.shape}")
print(f"\nDtypes:\n{income.dtypes}")
income.head(5)

## 2. Market Data Ingestion

In [ ]:
from src.ingest.market import run as run_market

prices = run_market()

In [ ]:
# Inspect prices_daily
prices = pd.read_parquet(PROCESSED_DIR / "prices_daily.parquet")
print(f"Shape: {prices.shape}")
print(f"\nDtypes:\n{prices.dtypes}")
print(f"\nTickers: {sorted(prices['ticker'].unique())}")
print(f"Date range: {prices['date'].min()} to {prices['date'].max()}")
prices.head(5)

## 3. Macro Data Ingestion

In [ ]:
from src.ingest.macro import run as run_macro

macro = run_macro()

In [ ]:
# Inspect macro_monthly
macro = pd.read_parquet(PROCESSED_DIR / "macro_monthly.parquet")
print(f"Shape: {macro.shape}")
print(f"\nDtypes:\n{macro.dtypes}")
macro.head(5)

---
## 4. Visualizations

### 4a. Price History — Normalized to Base 100

In [ ]:
prices = pd.read_parquet(PROCESSED_DIR / "prices_daily.parquet")

# Exclude benchmark for this plot
stock_prices = prices[prices["ticker"] != "IBOV"].copy()

# Normalize: first valid close = 100
def normalize(group):
    group = group.sort_values("date")
    first_close = group["close"].iloc[0]
    group["normalized"] = (group["close"] / first_close) * 100
    return group

stock_prices = stock_prices.groupby("ticker", group_keys=False).apply(normalize)

fig, ax = plt.subplots(figsize=(14, 7))
for ticker, grp in stock_prices.groupby("ticker"):
    ax.plot(grp["date"], grp["normalized"], label=ticker, linewidth=1)

ax.set_title("Price History — Normalized to Base 100 (2020–2025)", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Normalized Price (Base 100)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4b. Selic vs IPCA (Monthly)

In [ ]:
macro = pd.read_parquet(PROCESSED_DIR / "macro_monthly.parquet")

fig, ax1 = plt.subplots(figsize=(14, 6))

color_selic = "#1f77b4"
color_ipca = "#d62728"

ax1.set_xlabel("Date")
ax1.set_ylabel("Selic (% a.d.)", color=color_selic)
ax1.plot(macro["date"], macro["selic_daily"], color=color_selic, linewidth=1.5, label="Selic")
ax1.tick_params(axis="y", labelcolor=color_selic)

ax2 = ax1.twinx()
ax2.set_ylabel("IPCA (% mensal)", color=color_ipca)
ax2.plot(macro["date"], macro["ipca_monthly"], color=color_ipca, linewidth=1.5, label="IPCA")
ax2.tick_params(axis="y", labelcolor=color_ipca)

fig.suptitle("Selic vs IPCA — 2020–2025", fontsize=14)
fig.tight_layout()

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.show()

---
## 5. Data Quality Summary

In [ ]:
tables = {
    "fundamentals_long": pd.read_parquet(PROCESSED_DIR / "fundamentals_long.parquet"),
    "income_long": pd.read_parquet(PROCESSED_DIR / "income_long.parquet"),
    "prices_daily": pd.read_parquet(PROCESSED_DIR / "prices_daily.parquet"),
    "macro_monthly": pd.read_parquet(PROCESSED_DIR / "macro_monthly.parquet"),
}

print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

for name, df in tables.items():
    print(f"\n📊 {name}")
    print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    missing = df.isnull().sum()
    if missing.sum() == 0:
        print("   Missing values: NONE ✅")
    else:
        print("   Missing values per column:")
        for col, count in missing.items():
            pct = count / len(df) * 100
            status = "⚠️" if count > 0 else "✅"
            print(f"     {col}: {count} ({pct:.1f}%) {status}")

print("\n" + "=" * 60)
print("Pipeline complete!")
print("=" * 60)